# Hybrid Vector Search and Knowledge Graph Retrieval for Clinical Question Answering

Reproduction notebook for the experiments reported in the paper.

**Benchmarks**
- MedQA-USMLE — 1,273 test questions, Claude Opus 5
- PathVQA — 1,010 yes/no test questions, Claude Haiku

**Architectures**
- A. Baseline (no retrieval)
- B. Flat-vector RAG (FAISS)
- C. Static GraphRAG (NetworkX + PrimeKG)
- D. Hybrid GraphRAG (BM25 + FAISS + graph)
- E. Multimodal hybrid (vision + BM25 + VQA graph)

---

### Important: matched decoding settings

All architectures **within a benchmark** must use identical API settings. MedQA uses
`max_tokens=2000` with the model's default reasoning configuration; PathVQA uses
`max_tokens=10`.

This is not cosmetic. Evaluating the MedQA baseline with a reduced output limit and
reasoning disabled, while the retrieval architectures kept the default configuration,
lowered the baseline from 96.0% to 94.7% and made all three retrieval systems appear
significantly better. The effect was an artifact of decoding configuration, not retrieval.

### API keys

Keys are read from Colab Secrets (key icon in the left sidebar). Add a secret named
`ANTHROPIC_API_KEY`. Never hardcode a key in this notebook.

### Reproducibility

Sampling parameters are left at their API defaults, so re-running the
experiments produces statistically equivalent but not bit-identical output.
The CSVs in `results/` are the record of the runs reported in the paper.
Section 8 reproduces every number in Tables I and II from those files exactly,
with no API calls.

## 1. Setup

In [ ]:
!pip install anthropic datasets sentence-transformers faiss-cpu rank_bm25 statsmodels networkx -q

In [ ]:
import os, re, json, time, pickle, shutil, base64
from io import BytesIO

import numpy as np
import pandas as pd
import networkx as nx
import faiss
import anthropic
from datasets import load_dataset
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

from google.colab import drive, userdata

drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/graphrag_research/'
os.makedirs(DRIVE, exist_ok=True)

client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

MODEL_TEXT   = "claude-opus-5"
MODEL_VISION = "claude-haiku-4-5-20251001"
MAX_TOKENS_TEXT   = 2000   # matched across all MedQA architectures
MAX_TOKENS_VISION = 10     # matched across all PathVQA architectures

embedder = SentenceTransformer('all-MiniLM-L6-v2')
print("setup complete")

## 2. Datasets

MedQA-USMLE: 10,178 train / 1,273 test.
PathVQA: 19,654 train / 6,719 test; we evaluate the yes/no subset.

In [ ]:
medqa   = load_dataset("GBaker/MedQA-USMLE-4-options")
pathvqa = load_dataset("flaviagiammarino/path-vqa")

medqa_test = medqa['test']
pathvqa_yesno = [x for x in pathvqa['test'] if x['answer'].lower() in ('yes','no')]

print(f"MedQA test      : {len(medqa_test)}")
print(f"PathVQA yes/no  : {len(pathvqa_yesno)}")

## 3. Utilities

A single checkpointed evaluation loop is reused by every experiment. Results are written
to Drive every 10 questions so a disconnected runtime never loses more than a few.

In [ ]:
def call_text(prompt, retries=5):
    """Text-only call. Settings matched across all MedQA architectures."""
    for attempt in range(retries):
        try:
            r = client.messages.create(
                model=MODEL_TEXT,
                max_tokens=MAX_TOKENS_TEXT,
                messages=[{"role": "user", "content": prompt}])
            for b in r.content:
                if b.type == "text":
                    for ch in b.text.strip():
                        if ch in "ABCD":
                            return ch
            return "X"
        except Exception as e:
            print(f"  attempt {attempt+1}: {e}")
            time.sleep(60 if '529' in str(e) or 'overload' in str(e).lower() else 10)
    return "X"


def call_vision(image, prompt, retries=5):
    """Vision call. Settings matched across all PathVQA architectures."""
    buf = BytesIO(); image.convert('RGB').save(buf, format="JPEG")
    b64 = base64.standard_b64encode(buf.getvalue()).decode()
    for attempt in range(retries):
        try:
            r = client.messages.create(
                model=MODEL_VISION,
                max_tokens=MAX_TOKENS_VISION,
                messages=[{"role": "user", "content": [
                    {"type": "image", "source": {"type": "base64",
                     "media_type": "image/jpeg", "data": b64}},
                    {"type": "text", "text": prompt}]}])
            for b in r.content:
                if b.type == "text":
                    t = b.text.strip().lower()
                    if 'yes' in t: return 'yes'
                    if 'no'  in t: return 'no'
            return "unknown"
        except Exception as e:
            print(f"  attempt {attempt+1}: {e}")
            time.sleep(60 if '529' in str(e) or 'overload' in str(e).lower() else 10)
    return "unknown"


def run_eval(name, items, predict_fn, gold_fn, filename):
    """Checkpointed evaluation loop. Resumes automatically from Drive."""
    local, remote = filename, DRIVE + filename
    if os.path.exists(remote) and not os.path.exists(local):
        shutil.copy(remote, local)

    rows  = pd.read_csv(local).to_dict('records') if os.path.exists(local) else []
    start = len(rows)
    print(f"{name}: starting at {start+1}/{len(items)}")

    for i in range(start, len(items)):
        item = items[i]
        pred = predict_fn(item)
        gold = gold_fn(item)
        rows.append({'idx': i, 'predicted': pred, 'correct': gold,
                     'is_correct': pred == gold})
        if (i + 1) % 10 == 0:
            pd.DataFrame(rows).to_csv(local, index=False)
            shutil.copy(local, remote)
        acc = sum(r['is_correct'] for r in rows) / len(rows) * 100
        print(f"  {i+1}/{len(items)} | {pred} vs {gold} | {acc:.1f}%")
        time.sleep(0.5)

    df = pd.DataFrame(rows)
    df.to_csv(local, index=False); shutil.copy(local, remote)
    print(f"{name}: {df['is_correct'].mean()*100:.1f}%")
    return df

## 4. Knowledge graphs

Four graphs are built. The two LLM-extracted graphs cost API calls; PrimeKG and the
PathVQA graph are free. Each is cached to Drive.

In [ ]:
def extract_triples(text, retries=3):
    """LLM-driven entity and relation extraction, returns JSON."""
    prompt = (
        'Extract medical entities and relationships from this clinical text.\n'
        'Return ONLY a JSON object. No explanation, no markdown.\n\n'
        'Format:\n{"entities": [{"name": "X", "type": '
        '"disease|symptom|medication|treatment|procedure"}], '
        '"relationships": [{"source": "X", "target": "Y", "relation": '
        '"treats|causes|indicates|contraindicates"}]}\n\n'
        f'Text: {text[:400]}\n\nJSON:')
    for attempt in range(retries):
        try:
            r = client.messages.create(model=MODEL_TEXT, max_tokens=600,
                                       messages=[{"role":"user","content":prompt}])
            out = next((b.text.strip() for b in r.content if b.type == "text"), "")
            out = out.replace("```json","").replace("```","").strip()
            m = re.search(r'\{.*\}', out, re.DOTALL)
            if m: out = m.group()
            out = re.sub(r',\s*}', '}', out)
            out = re.sub(r',\s*]', ']', out)
            return json.loads(out)
        except Exception as e:
            print(f"  attempt {attempt+1}: {e}")
            time.sleep(3)
    return {"entities": [], "relationships": []}


def build_llm_graph(docs, path, n_docs):
    """Build a typed graph from documents via LLM extraction. Cached to Drive."""
    if os.path.exists(DRIVE + path):
        G = pickle.load(open(DRIVE + path, 'rb'))
        print(f"loaded {path}: {G.number_of_nodes()} entities, {G.number_of_edges()} relations")
        return G

    G = nx.Graph()
    for i in range(n_docs):
        ex = extract_triples(docs[i])
        for e in ex.get("entities", []):
            G.add_node(e["name"], type=e["type"])
        for r in ex.get("relationships", []):
            s, t = r.get("source"), r.get("target")
            if s and t and s in G and t in G:
                G.add_edge(s, t, relation=r.get("relation", "related_to"))
        if (i + 1) % 25 == 0:
            pickle.dump(G, open(DRIVE + path, 'wb'))
            print(f"  {i+1}/{n_docs} | {G.number_of_nodes()} entities")
        time.sleep(0.5)

    pickle.dump(G, open(DRIVE + path, 'wb'))
    print(f"{path}: {G.number_of_nodes()} entities, {G.number_of_edges()} relations")
    return G

In [ ]:
# --- MedQA graph: 500 training documents -> 1,330 entities, 1,190 relations ---
medqa_docs = [f"{x['question']} Answer: {x['answer']}" for x in medqa['train'].select(range(500))]
G_medqa = build_llm_graph(medqa_docs, 'knowledge_graph_500.pkl', 500)

In [ ]:
# --- PubMedQA graph: 500 abstracts -> 1,968 entities, 1,549 relations ---
pubmedqa = load_dataset("qiaojin/PubMedQA", "pqa_labeled")
pubmed_docs = [f"{x['question']} {' '.join(x['context']['contexts'][:2])} Answer: {x['long_answer']}"
               for x in pubmedqa['train'].select(range(500))]
G_pubmed = build_llm_graph(pubmed_docs, 'knowledge_graph_pubmed.pkl', 500)

In [ ]:
# --- PrimeKG: Harvard precision medicine graph, filtered to clinical relations ---
# Full graph: 129,375 nodes / 4,050,249 relations. Filtered here to 12,752 / 441,429.
PRIME = 'knowledge_graph_primekg.pkl'

if os.path.exists(DRIVE + PRIME):
    G_prime = pickle.load(open(DRIVE + PRIME, 'rb'))
else:
    !wget -q -O kg.csv https://dataverse.harvard.edu/api/access/datafile/6180620
    primekg = pd.read_csv('kg.csv', low_memory=False)
    clinical = primekg[
        (primekg['x_type'].isin(['disease','drug'])) |
        (primekg['y_type'].isin(['disease','drug']))
    ].head(500000)

    G_prime = nx.Graph()
    for _, r in clinical.iterrows():
        G_prime.add_node(r['x_name'], type=r['x_type'])
        G_prime.add_node(r['y_name'], type=r['y_type'])
        G_prime.add_edge(r['x_name'], r['y_name'], relation=r['display_relation'])
    pickle.dump(G_prime, open(DRIVE + PRIME, 'wb'))

print(f"PrimeKG: {G_prime.number_of_nodes()} entities, {G_prime.number_of_edges()} relations")

In [ ]:
# --- PathVQA graph: rule-based extraction from 19,654 training examples ---
VQA_G = 'knowledge_graph_vqa.pkl'
STOP = {'are','the','is','there','what','which','this','that','for','and','has',
        'have','can','any','with','from','been','does','shown','image','present','seen'}

if os.path.exists(DRIVE + VQA_G):
    G_vqa = pickle.load(open(DRIVE + VQA_G, 'rb'))
else:
    G_vqa = nx.Graph()
    for x in pathvqa['train']:
        terms = [w for w in re.findall(r'\b[a-z]{3,}\b', x['question'].lower()) if w not in STOP]
        ans = x['answer'].lower()
        for t in terms:
            G_vqa.add_node(t, type='medical_concept')
        if ans in ('yes','no'):
            for t in terms: G_vqa.add_edge(t, ans, relation='finding')
        else:
            G_vqa.add_node(ans, type='finding')
            for t in terms: G_vqa.add_edge(t, ans, relation='indicates')
    pickle.dump(G_vqa, open(DRIVE + VQA_G, 'wb'))

print(f"VQA graph: {G_vqa.number_of_nodes()} entities, {G_vqa.number_of_edges()} relations")

## 5. Retrieval indexes

In [ ]:
def graph_context(question, G, node_list, node_emb, top_k=5):
    """Retrieve a subgraph as formatted triples (Algorithm 1)."""
    q = embedder.encode([question])
    sims = np.dot(node_emb, q.T).flatten()
    out = []
    for idx in np.argsort(sims)[-top_k:][::-1]:
        n = node_list[idx]
        out.append(f"Entity: {n} ({G.nodes[n].get('type','unknown')})")
        for nb in list(G.neighbors(n))[:3]:
            rel = G.edges[n, nb].get('relation', 'related_to')
            out.append(f"  -> {n} {rel} {nb}")
    return "\n".join(out)


def build_indexes(docs):
    """BM25 (Algorithm 2) and FAISS dense index over the same corpus."""
    bm25 = BM25Okapi([d.lower().split() for d in docs])
    emb  = embedder.encode(docs, show_progress_bar=True, batch_size=64)
    index = faiss.IndexFlatL2(emb.shape[1])
    index.add(emb.astype('float32'))
    return bm25, index

In [ ]:
# MedQA corpus: full 10,178-document training set
medqa_corpus = [f"Question: {x['question']}\nCorrect Answer: {x['answer']}" for x in medqa['train']]
bm25_medqa, faiss_medqa = build_indexes(medqa_corpus)

prime_nodes = list(G_prime.nodes())
prime_emb   = embedder.encode(prime_nodes, show_progress_bar=True, batch_size=64)

# PathVQA corpus: 19,654 training examples
vqa_corpus = [f"Q: {x['question']} A: {x['answer']}" for x in pathvqa['train']]
bm25_vqa, faiss_vqa = build_indexes(vqa_corpus)

vqa_nodes = list(G_vqa.nodes())
vqa_emb   = embedder.encode(vqa_nodes, show_progress_bar=True, batch_size=64)
print("indexes ready")

## 6. MedQA experiments

Four architectures, 1,273 questions each. All use identical decoding settings.

In [ ]:
def opts_block(options):
    return "\n".join(f"{k}: {v}" for k, v in options.items())

MEDQA_TAIL = "\n\nReply with only the letter A, B, C, or D. Nothing else."


def medqa_baseline(x):
    return call_text(f"You are a medical expert. Answer this clinical question.\n\n"
                     f"Question: {x['question']}\n\n{opts_block(x['options'])}{MEDQA_TAIL}")


def medqa_rag(x):
    q = embedder.encode([x['question']])
    _, idx = faiss_medqa.search(q.astype('float32'), 3)
    ctx = "\n\n".join(medqa_corpus[i] for i in idx[0])
    return call_text(f"You are a medical expert. Use the following reference cases.\n\n"
                     f"Reference Cases:\n{ctx}\n\nQuestion: {x['question']}\n\n"
                     f"{opts_block(x['options'])}{MEDQA_TAIL}")


def medqa_graphrag(x):
    ctx = graph_context(x['question'], G_prime, prime_nodes, prime_emb)
    return call_text(f"You are a medical expert. Use the knowledge graph context.\n\n"
                     f"Knowledge Graph:\n{ctx}\n\nQuestion: {x['question']}\n\n"
                     f"{opts_block(x['options'])}{MEDQA_TAIL}")


def medqa_hybrid(x):
    bm  = np.argsort(bm25_medqa.get_scores(x['question'].lower().split()))[-3:][::-1]
    bm_ctx = "\n\n".join(medqa_corpus[i] for i in bm)
    q = embedder.encode([x['question']])
    _, idx = faiss_medqa.search(q.astype('float32'), 3)
    vec_ctx = "\n\n".join(medqa_corpus[i] for i in idx[0])
    g_ctx = graph_context(x['question'], G_prime, prime_nodes, prime_emb)
    return call_text(f"You are a medical expert. Use the following evidence.\n\n"
                     f"BM25 Retrieved Cases:\n{bm_ctx}\n\n"
                     f"Vector Retrieved Cases:\n{vec_ctx}\n\n"
                     f"Clinical Knowledge Graph:\n{g_ctx}\n\n"
                     f"Question: {x['question']}\n\n{opts_block(x['options'])}{MEDQA_TAIL}")

In [ ]:
gold_medqa = lambda x: x['answer_idx']

df_med_base   = run_eval("MedQA baseline", medqa_test, medqa_baseline, gold_medqa, 'baseline_medqa_matched.csv')
df_med_rag    = run_eval("MedQA RAG",      medqa_test, medqa_rag,      gold_medqa, 'rag_opus5_results.csv')
df_med_graph  = run_eval("MedQA GraphRAG", medqa_test, medqa_graphrag, gold_medqa, 'graphrag_opus5_results.csv')
df_med_hybrid = run_eval("MedQA Hybrid",   medqa_test, medqa_hybrid,   gold_medqa, 'hybrid_graphrag_results.csv')

## 7. PathVQA experiments

Four architectures, 1,010 yes/no questions each, Claude Haiku with vision.

In [ ]:
VQA_TAIL = "\n\nReply with only: yes or no. Nothing else."


def vqa_baseline(x):
    return call_vision(x['image'],
        f"You are a medical pathology expert. Analyze this image and answer.\n\n"
        f"Question: {x['question']}{VQA_TAIL}")


def vqa_rag(x):
    q = embedder.encode([x['question']])
    _, idx = faiss_vqa.search(q.astype('float32'), 3)
    ctx = "\n".join(vqa_corpus[i] for i in idx[0])
    return call_vision(x['image'],
        f"You are a medical pathology expert. Use these similar cases.\n\n"
        f"Similar Cases:\n{ctx}\n\nQuestion: {x['question']}{VQA_TAIL}")


def vqa_graphrag(x):
    ctx = graph_context(x['question'], G_vqa, vqa_nodes, vqa_emb)
    return call_vision(x['image'],
        f"You are a medical pathology expert. Use the knowledge graph context.\n\n"
        f"Knowledge Graph:\n{ctx}\n\nQuestion: {x['question']}{VQA_TAIL}")


def vqa_hybrid(x):
    bm = np.argsort(bm25_vqa.get_scores(x['question'].lower().split()))[-5:][::-1]
    bm_ctx = "\n".join(vqa_corpus[i] for i in bm)
    g_ctx  = graph_context(x['question'], G_vqa, vqa_nodes, vqa_emb)
    return call_vision(x['image'],
        f"You are a medical pathology expert. Use the image, similar cases and knowledge graph.\n\n"
        f"Similar Cases:\n{bm_ctx}\n\nKnowledge Graph:\n{g_ctx}\n\n"
        f"Question: {x['question']}{VQA_TAIL}")

In [ ]:
gold_vqa = lambda x: x['answer'].lower()

# The three PathVQA architectures reported in the paper were run together and stored
# in a single combined file (columns: baseline, graphrag, hybrid). If that file is
# present it is loaded directly; otherwise each architecture is evaluated from scratch.
COMBINED = DRIVE + 'pathvqa_results.csv'

if os.path.exists(COMBINED):
    pv = pd.read_csv(COMBINED)
    print(f"loaded combined PathVQA results: {len(pv)} questions")
    def _col(name):
        return pd.DataFrame({'idx': range(len(pv)),
                             'is_correct': pv[name].astype(bool)})
    df_vqa_base  = _col('baseline')
    df_vqa_graph = _col('graphrag')
    df_vqa_hybrid = _col('hybrid')
    for label, df in [('baseline', df_vqa_base), ('GraphRAG', df_vqa_graph),
                      ('hybrid', df_vqa_hybrid)]:
        print(f"  {label:<9} {df['is_correct'].mean()*100:.1f}%")
else:
    df_vqa_base   = run_eval("PathVQA baseline", pathvqa_yesno, vqa_baseline,  gold_vqa, 'pathvqa_baseline.csv')
    df_vqa_graph  = run_eval("PathVQA GraphRAG", pathvqa_yesno, vqa_graphrag,  gold_vqa, 'pathvqa_graphrag.csv')
    df_vqa_hybrid = run_eval("PathVQA Hybrid",   pathvqa_yesno, vqa_hybrid,    gold_vqa, 'pathvqa_hybrid.csv')

# Flat-vector RAG was run separately and is stored in its own file.
df_vqa_rag = run_eval("PathVQA RAG", pathvqa_yesno, vqa_rag, gold_vqa, 'pathvqa_rag_results.csv')

## 8. Statistical analysis

McNemar's test on paired binary outcomes, plus 95% Wilson confidence intervals.
The exact binomial test is used when discordant pairs are few.

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.proportion import proportion_confint


def analyse(title, systems):
    n = min(len(v) for v in systems.values())
    systems = {k: v[:n] for k, v in systems.items()}
    print("=" * 68)
    print(f"{title}  (n = {n}, paired)")
    print("=" * 68)

    print(f"\n{'System':<20}{'Correct':>9}{'Acc':>9}   95% Wilson CI")
    print("-" * 68)
    for k, v in systems.items():
        c = int(v.sum())
        lo, hi = proportion_confint(c, n, 0.05, method='wilson')
        print(f"{k:<20}{c:>9}{c/n*100:>8.1f}%   [{lo*100:.1f}%, {hi*100:.1f}%]")

    base = systems['Baseline']
    print(f"\n{'Comparison vs Baseline':<34}{'b':>5}{'c':>5}{'p':>10}   Verdict")
    print("-" * 68)
    for k, v in systems.items():
        if k == 'Baseline':
            continue
        b = int(np.sum(base & ~v)); c = int(np.sum(~base & v))
        exact = (b + c) < 25
        p = mcnemar([[0, b], [c, 0]], exact=exact, correction=not exact).pvalue
        print(f"{k+' vs Baseline':<34}{b:>5}{c:>5}{p:>10.4f}   "
              f"{'significant' if p < 0.05 else 'not significant'}")
    print()


def vec(df): return df['is_correct'].astype(bool).values

analyse("MedQA-USMLE", {
    'Baseline':        vec(df_med_base),
    'Flat-Vector RAG': vec(df_med_rag),
    'Static GraphRAG': vec(df_med_graph),
    'Hybrid GraphRAG': vec(df_med_hybrid)})

analyse("PathVQA", {
    'Baseline':        vec(df_vqa_base),
    'Flat-Vector RAG': vec(df_vqa_rag),
    'Static GraphRAG': vec(df_vqa_graph),
    'Hybrid GraphRAG': vec(df_vqa_hybrid)})

## 9. Figures

Regenerates Figure 3 from the result CSVs. Figure 2 is a static architecture diagram
and is included in the repository as a PNG.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

def figure3(medqa_acc, pathvqa_acc, medqa_n, pathvqa_n, outfile='figure3_results.png'):
    """Two-panel accuracy comparison. Values are read from the analysis above."""
    systems = ['Baseline\nLLM', 'Flat-Vector\nRAG', 'Static\nGraphRAG', 'Hybrid\nGraphRAG']
    GREY, TEAL, PURPLE = '#4A4A4A', '#1D7A6B', '#5B4DB7'
    colors = [GREY, TEAL, TEAL, PURPLE]

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle('Retrieval Architecture Performance Across Clinical QA Benchmarks',
                 fontsize=13, fontweight='bold', y=1.02)

    # left: MedQA
    bars = axes[0].bar(systems, medqa_acc, color=colors, edgecolor='white', width=0.5)
    for bar, v in zip(bars, medqa_acc):
        axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                     f'{v}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
    axes[0].set_title(f'MedQA-USMLE (n = {medqa_n:,})\n'
                      'all differences n.s. vs baseline (p = 0.34-0.72)',
                      fontsize=11, fontweight='bold', pad=10)
    axes[0].set_ylim(min(medqa_acc)-1.3, max(medqa_acc)+1.0)
    axes[0].axhline(medqa_acc[0], color=GREY, ls='--', lw=1.2, alpha=0.7,
                    label=f'Baseline ({medqa_acc[0]}%)')

    # right: PathVQA
    bars = axes[1].bar(systems, pathvqa_acc, color=colors, edgecolor='white', width=0.5)
    for bar, v in zip(bars, pathvqa_acc):
        axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                     f'{v}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
    axes[1].set_title(f'PathVQA (n = {pathvqa_n:,})', fontsize=11, fontweight='bold', pad=10)
    axes[1].set_ylim(min(pathvqa_acc)-6, max(pathvqa_acc)+5)
    axes[1].axhline(pathvqa_acc[0], color=GREY, ls='--', lw=1.2, alpha=0.7,
                    label=f'Baseline ({pathvqa_acc[0]}%)')
    axes[1].text(1, pathvqa_acc[1]+2.0, 'n.s.\n(p = 0.21)', ha='center',
                 fontsize=8, color='#777777')
    axes[1].text(2, pathvqa_acc[2]+2.0, 'n.s.\n(p = 0.26)', ha='center',
                 fontsize=8, color='#777777')
    delta = round(pathvqa_acc[3] - pathvqa_acc[0], 1)
    axes[1].annotate(f'+{delta} pp, p < 0.001',
                     xy=(3, pathvqa_acc[3]), xytext=(2.05, pathvqa_acc[3]+3.0),
                     fontsize=8.5, color=PURPLE, fontweight='bold',
                     arrowprops=dict(arrowstyle='->', color=PURPLE, lw=0.9))

    for ax in axes:
        ax.set_ylabel('Exact-Match Accuracy (%)', fontsize=10)
        ax.legend(fontsize=9, loc='lower right')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.tick_params(labelsize=9)
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.1f}%'))

    fig.legend(handles=[Patch(facecolor=GREY,   label='Baseline (no retrieval)'),
                        Patch(facecolor=TEAL,   label='Single-strategy retrieval (n.s.)'),
                        Patch(facecolor=PURPLE, label='Hybrid GraphRAG')],
               loc='lower center', ncol=3, fontsize=9, bbox_to_anchor=(0.5, -0.05))
    plt.tight_layout()
    plt.savefig(outfile, dpi=300, bbox_inches='tight', facecolor='white')
    plt.savefig(DRIVE + outfile, dpi=300, bbox_inches='tight', facecolor='white')
    print(f'saved {outfile}')


def acc(df):
    return round(df['is_correct'].mean() * 100, 1)

figure3(
    medqa_acc   = [acc(df_med_base), acc(df_med_rag), acc(df_med_graph), acc(df_med_hybrid)],
    pathvqa_acc = [acc(df_vqa_base), acc(df_vqa_rag), acc(df_vqa_graph), acc(df_vqa_hybrid)],
    medqa_n     = len(df_med_base),
    pathvqa_n   = len(df_vqa_base))

## Appendix: decoding-configuration control

The paper reports that evaluating the MedQA baseline with a reduced output limit and
reasoning disabled, while the retrieval architectures kept the default configuration,
lowered the baseline from 96.0% to 94.7% and made all three retrieval systems appear
significantly better.

This cell reproduces that mismatched run so the effect can be verified. **It is a
control, not a result** — the mismatched baseline must not be compared against the
retrieval architectures in any reported analysis.

In [ ]:
def medqa_baseline_no_thinking(x, retries=5):
    """Deliberately mismatched: reduced output limit and reasoning disabled."""
    prompt = (f"You are a medical expert. Answer this clinical question.\n\n"
              f"Question: {x['question']}\n\n{opts_block(x['options'])}{MEDQA_TAIL}")
    for attempt in range(retries):
        try:
            r = client.messages.create(
                model=MODEL_TEXT,
                max_tokens=10,                        # mismatched (paper uses 2000)
                thinking={"type": "disabled"},        # mismatched (paper uses default)
                messages=[{"role": "user", "content": prompt}])
            for b in r.content:
                if b.type == "text":
                    for ch in b.text.strip():
                        if ch in "ABCD":
                            return ch
            return "X"
        except Exception as e:
            print(f"  attempt {attempt+1}: {e}")
            time.sleep(60 if '529' in str(e) else 10)
    return "X"


# Uncomment to reproduce. Expect ~94.7% versus 96.0% for the matched baseline.
# df_med_nothink = run_eval("MedQA baseline (no thinking)", medqa_test,
#                           medqa_baseline_no_thinking, gold_medqa,
#                           'baseline_medqa_nothinking.csv')
#
# analyse("MedQA with MISMATCHED baseline -- illustrative only", {
#     'Baseline':        vec(df_med_nothink),
#     'Flat-Vector RAG': vec(df_med_rag),
#     'Static GraphRAG': vec(df_med_graph),
#     'Hybrid GraphRAG': vec(df_med_hybrid)})

## Results as reported in the paper

| System | MedQA (n=1,273) | PathVQA (n=1,010) |
|---|---|---|
| Baseline | 96.0% | 68.0% |
| Flat-Vector RAG | 96.5% (p=0.34) | 70.4% (p=0.21) |
| Static GraphRAG | 95.8% (p=0.72) | 69.9% (p=0.26) |
| **Hybrid GraphRAG** | **96.2% (p=0.72)** | **79.6% (p<0.001)** |

On MedQA no architecture differs significantly from the baseline — the benchmark is
saturated for a frontier model. On PathVQA only the hybrid system clears significance;
neither flat-vector RAG nor static GraphRAG improves on the baseline alone.